# Fuzzy time-series forecasting: a multi-horizon experiment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/D2718281828nis/Fuzzy-Regression/blob/master/example-Fuzzy-timeseries-forecast.ipynb)

This notebook implements frequency-weighted fuzzy time series (FTS) from first principles and compares them with transparent baselines. We evaluate **1-, 2-, and 3-step-ahead** forecasts with a leakage-free rolling origin.

> **Question.** Do first- and second-order fuzzy rules provide useful forecasts, and how does their accuracy change with the forecast horizon?

In [ ]:
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=2, suppress=True)

## 1. Data and experimental design

The classic University of Alabama enrollment series contains only 22 annual observations. We therefore avoid a single hand-picked split. At every rolling origin, all model parameters—including the fuzzy universe and rules—are re-estimated using the available prefix only. The following 1–3 observations remain hidden for testing.

In [ ]:
years = np.arange(1971, 1993)
values = np.array([
    13055, 13563, 13867, 14696, 15460, 15311, 15603, 15861,
    16807, 16919, 16388, 15433, 15497, 15145, 15163, 15984,
    16859, 18150, 18970, 19328, 19337, 18876,
], dtype=float)
series = pd.Series(values, index=years, name="enrollment")

ax = series.plot(marker="o", figsize=(10, 4), title="University of Alabama enrollment")
ax.set(xlabel="Year", ylabel="Students")
plt.show()
series.to_frame().tail()

## 2. Frequency-weighted FTS

We pad the training range by 10%, divide it into equal intervals, and represent each interval by its midpoint. Assigning an observation to the nearest midpoint is the maximum-membership choice for regular overlapping triangular sets.

A model of order 1 learns `A(t-1) -> A(t)`; order 2 learns `(A(t-2), A(t-1)) -> A(t)`. Consequents retain their observed counts. Defuzzification is their frequency-weighted midpoint. The order-2 model backs off to its first-order table for an unseen pair. Forecasts are recursive, so no actual future value is used after the origin.

In [ ]:
class WeightedFTS:
    """Frequency-weighted first- or second-order fuzzy time-series model."""

    def __init__(self, n_intervals=7, order=1, padding=0.10):
        if order not in (1, 2):
            raise ValueError("order must be 1 or 2")
        self.n_intervals = n_intervals
        self.order = order
        self.padding = padding

    def fit(self, y):
        y = np.asarray(y, dtype=float)
        if y.size <= self.order:
            raise ValueError("The training series is too short for this order")
        span = np.ptp(y) or max(abs(y[0]), 1.0)
        lo, hi = y.min() - self.padding * span, y.max() + self.padding * span
        edges = np.linspace(lo, hi, self.n_intervals + 1)
        self.centers_ = (edges[:-1] + edges[1:]) / 2
        states = self._fuzzify(y)
        self.rules_ = self._make_rules(states, self.order)
        self.backoff_rules_ = self._make_rules(states, 1)
        self.history_ = list(y)
        return self

    def _fuzzify(self, x):
        x = np.atleast_1d(x).astype(float)
        return np.abs(x[:, None] - self.centers_[None, :]).argmin(axis=1)

    @staticmethod
    def _make_rules(states, order):
        rules = defaultdict(Counter)
        for t in range(order, len(states)):
            context = tuple(states[t-order:t])
            rules[context][states[t]] += 1
        return dict(rules)

    def _next(self, history):
        states = self._fuzzify(history)
        context = tuple(states[-self.order:])
        counts = self.rules_.get(context)
        if not counts and self.order == 2:
            counts = self.backoff_rules_.get((states[-1],))
        if not counts:
            return float(self.centers_[states[-1]])
        next_states = np.fromiter(counts.keys(), dtype=int)
        weights = np.fromiter(counts.values(), dtype=float)
        return float(np.average(self.centers_[next_states], weights=weights))

    def predict(self, steps=1):
        history, forecasts = self.history_.copy(), []
        for _ in range(steps):
            forecast = self._next(history)
            forecasts.append(forecast)
            history.append(forecast)
        return np.array(forecasts)

### Inspect one fitted rule base

The cell below makes the linguistic mechanism visible. State labels are interval numbers; counts show how often each consequent followed its context in the full series.

In [ ]:
illustration = WeightedFTS(n_intervals=7, order=1).fit(values)
rule_rows = []
for context, consequents in sorted(illustration.rules_.items()):
    rule_rows.append({
        "from": f"A{context[0] + 1}",
        "to (frequency)": ", ".join(
            f"A{state + 1} ({count})" for state, count in sorted(consequents.items())
        ),
    })
pd.DataFrame(rule_rows)

## 3. Competing approaches

- **Naive / persistence:** repeat the last observation; a hard-to-beat benchmark for short series.
- **Linear trend:** least-squares regression on time, directly evaluated at each future index.
- **Weighted FTS(1)** and **Weighted FTS(2):** recursive fuzzy forecasts with 7 intervals.

Seven intervals are fixed before evaluation rather than tuned on the test values. This is a demonstration, not a claim that seven is optimal.

In [ ]:
def forecast_methods(train, steps):
    train = np.asarray(train, dtype=float)
    x = np.arange(train.size).reshape(-1, 1)
    trend = LinearRegression().fit(x, train)
    future_x = np.arange(train.size, train.size + steps).reshape(-1, 1)
    return {
        "Naive": np.repeat(train[-1], steps),
        "Linear trend": trend.predict(future_x),
        "Weighted FTS(1)": WeightedFTS(7, order=1).fit(train).predict(steps),
        "Weighted FTS(2)": WeightedFTS(7, order=2).fit(train).predict(steps),
    }

forecast_methods(values[:-3], steps=3)

## 4. Rolling-origin evaluation

The first origin uses 10 training observations. Later origins expand the training window. Every origin must have all three future observations, giving the same number of errors per method at each horizon and making horizon comparisons straightforward.

In [ ]:
max_horizon = 3
min_train_size = 10
records = []

for train_end in range(min_train_size, len(values) - max_horizon + 1):
    train = values[:train_end]
    forecasts = forecast_methods(train, max_horizon)
    for method, prediction in forecasts.items():
        for horizon in range(1, max_horizon + 1):
            actual = values[train_end + horizon - 1]
            records.append({
                "origin": int(years[train_end - 1]),
                "target_year": int(years[train_end + horizon - 1]),
                "horizon": horizon,
                "method": method,
                "actual": actual,
                "forecast": prediction[horizon - 1],
            })

predictions = pd.DataFrame(records)
predictions.head(8)

In [ ]:
def metric_table(frame):
    rows = []
    for (method, horizon), group in frame.groupby(["method", "horizon"], sort=False):
        error = group["forecast"] - group["actual"]
        rows.append({
            "method": method,
            "horizon": horizon,
            "n": len(group),
            "MAE": np.abs(error).mean(),
            "RMSE": np.sqrt(np.square(error).mean()),
            "MAPE (%)": (np.abs(error) / group["actual"]).mean() * 100,
        })
    return pd.DataFrame(rows).sort_values(["horizon", "MAPE (%)"])

metrics = metric_table(predictions)
metrics.style.format({"MAE": "{:.1f}", "RMSE": "{:.1f}", "MAPE (%)": "{:.2f}"})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for horizon, ax in zip(range(1, 4), axes):
    view = metrics[metrics["horizon"] == horizon]
    ax.bar(view["method"], view["MAPE (%)"], color=["#4c78a8", "#f58518", "#54a24b", "#e45756"])
    ax.set_title(f"Horizon {horizon}")
    ax.set_xlabel("Method")
    ax.tick_params(axis="x", rotation=35)
axes[0].set_ylabel("MAPE (%)")
fig.suptitle("Rolling-origin error by forecast horizon")
fig.tight_layout()
plt.show()

## 5. Concrete 1-, 2-, and 3-year forecasts

Finally, fit each method through 1989 and predict the held-out years 1990–1992. This plot is an interpretable example; the rolling table above—not this single origin—is the more reliable comparison.

In [ ]:
cutoff = 1989
train_mask = years <= cutoff
example_forecasts = forecast_methods(values[train_mask], steps=3)
future_years = years[~train_mask]

example_table = pd.DataFrame({"Actual": values[~train_mask]}, index=future_years)
for method, forecast in example_forecasts.items():
    example_table[method] = forecast
example_table.round(1)

In [ ]:
ax = series.loc[1985:].plot(marker="o", color="black", figsize=(11, 5), label="Actual")
for method, forecast in example_forecasts.items():
    ax.plot(np.r_[cutoff, future_years], np.r_[series.loc[cutoff], forecast], marker="o", linestyle="--", label=method)
ax.axvline(cutoff, color="grey", linestyle=":", label="Forecast origin")
ax.set(title="Forecasts made at the end of 1989", xlabel="Year", ylabel="Students")
ax.legend(ncol=2)
plt.show()

## 6. Interpretation and limitations

1. Compare methods **within each horizon** in the metrics table. The ranking is empirical and can differ as the horizon grows.
2. FTS is interpretable: every estimate comes from observed fuzzy-state transitions. Weighting prevents a rare transition from counting as much as a frequent one.
3. Second order is not automatically better. With only 22 points, pairs of states are often unseen, so the model frequently needs its first-order fallback.
4. Recursive forecasts can collapse to a stable fuzzy state and accumulate discretization error. A linear trend extrapolates instead; persistence assumes no change. Their different inductive biases explain the different paths.
5. Results depend on interval count, partition scheme, and universe padding. Those are hyperparameters and should be selected inside a validation loop on a larger data set—not by looking at the final test years.
6. Enrollment is positive and far from zero, so MAPE is safe here. For series containing zeros, prefer MAE/RMSE or a suitable scaled metric.

The experiment therefore shows *how* the approaches behave rather than declaring a universal winner.